**Purpose**: Add table and column comments across all layers of the nyc_mobility catalog for Genie Space accuracy.

**Owner**: Course Author

**Inputs**: Existing tables in nyc_mobility (bronze, silver, gold).

**Outputs**: Six documented tables via COMMENT ON TABLE statements, plus column-level comments on high-value columns.

### `nyc_mobility.bronze.trips`
Raw NYC Yellow Taxi trip records from January 2024, ingested faithfully from the TLC monthly Parquet file with three operational metadata columns appended. No transformations applied; columns follow the TLC source schema and are documented at the table level only.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.bronze.trips IS 'Raw NYC Yellow Taxi trip records from January 2024, ingested faithfully from the TLC monthly Parquet file with three operational metadata columns appended: ingestion_timestamp, source_file, and pipeline_run_id. No transformations applied. Owner: 404Learning Data Engineering. Freshness: Refreshed when the TLC publishes a new monthly Parquet file, typically monthly with a two-month lag.';

### `nyc_mobility.bronze.taxi_zones`
TLC Taxi Zones reference table mapping LocationID values to zone name, borough, and service zone classification; used as the lookup source for Silver join enrichments. Note: the zone key column is physically named `location_id`.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.bronze.taxi_zones IS 'TLC Taxi Zones reference table mapping LocationID values to zone name, borough, and service zone classification. Used as the lookup source for Silver join enrichments. Owner: 404Learning Data Engineering. Freshness: Static reference data; refreshed only when TLC publishes an updated Taxi Zones CSV, typically once per year.';

In [0]:
%sql
COMMENT ON COLUMN nyc_mobility.bronze.taxi_zones.location_id IS 'Integer code that uniquely identifies a TLC taxi zone in the reference data.';
COMMENT ON COLUMN nyc_mobility.bronze.taxi_zones.Borough IS 'Name of the New York City borough containing the zone, from the TLC reference data.';
COMMENT ON COLUMN nyc_mobility.bronze.taxi_zones.Zone IS 'Human-readable name of the taxi zone, taken from the TLC reference data.';
COMMENT ON COLUMN nyc_mobility.bronze.taxi_zones.service_zone IS 'Service designation from the TLC reference data: Yellow Zone, Boro Zone, EWR, or N/A.';

### `nyc_mobility.silver.trips`
Cleaned and enriched NYC Yellow Taxi trips for January 2024: quality filters applied, Taxi Zones joined for pickup and dropoff names and boroughs, and four derived time-of-day columns. Column comments below cover derived and enriched columns only; Bronze pass-through columns follow the TLC source schema.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.silver.trips IS 'Cleaned and enriched NYC Yellow Taxi trips for January 2024 with quality filters applied (positive fare, positive distance, valid passenger count, dropoff after pickup), Taxi Zones joined for pickup and dropoff names and boroughs, and four derived time-of-day columns (trip_duration_minutes, pickup_hour, pickup_day_of_week, is_weekend). Owner: 404Learning Data Engineering. Freshness: Downstream of Bronze refresh; no independent schedule; rebuilt whenever Bronze updates.';

In [0]:
%sql
COMMENT ON COLUMN nyc_mobility.silver.trips.pickup_zone IS 'Zone name where the trip began, added by the Taxi Zones join on PULocationID.';
COMMENT ON COLUMN nyc_mobility.silver.trips.pickup_borough IS 'Borough where the trip began, added by the Taxi Zones join on PULocationID.';
COMMENT ON COLUMN nyc_mobility.silver.trips.dropoff_zone IS 'Zone name where the trip ended, added by the Taxi Zones join on DOLocationID.';
COMMENT ON COLUMN nyc_mobility.silver.trips.dropoff_borough IS 'Borough where the trip ended, added by the Taxi Zones join on DOLocationID.';
COMMENT ON COLUMN nyc_mobility.silver.trips.trip_duration_minutes IS 'Trip length in minutes, computed as the difference between dropoff and pickup timestamps.';
COMMENT ON COLUMN nyc_mobility.silver.trips.pickup_hour IS 'Hour of pickup in 24-hour format from 0 to 23, derived from tpep_pickup_datetime.';
COMMENT ON COLUMN nyc_mobility.silver.trips.pickup_day_of_week IS 'Day of week of pickup where 1 is Sunday and 7 is Saturday, derived from tpep_pickup_datetime.';
COMMENT ON COLUMN nyc_mobility.silver.trips.is_weekend IS 'Boolean flag that is true when the pickup day of week falls on Saturday or Sunday.';
COMMENT ON COLUMN nyc_mobility.silver.trips.RatecodeID IS 'TLC rate code: 1 Standard, 2 JFK flat, 3 Newark, 4 Nassau/Westchester, 5 Negotiated, 6 Group ride.';
COMMENT ON COLUMN nyc_mobility.silver.trips.payment_type IS 'TLC payment type: 1 Credit card, 2 Cash, 3 No charge, 4 Dispute, 5 Unknown, 6 Voided.';

### `nyc_mobility.gold.revenue_by_date`
Daily revenue aggregation of Silver trips, with total_revenue summed across total_amount and trip_count per pickup_date. Liquid clustered on pickup_date.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.gold.revenue_by_date IS 'Daily revenue aggregation of Silver trips, with total_revenue summed across total_amount and trip_count per pickup_date. Liquid clustered on pickup_date. Owner: 404Learning Data Engineering. Freshness: Downstream of Silver refresh; rebuilt as part of the Gold pipeline run.';

In [0]:
%sql
COMMENT ON COLUMN nyc_mobility.gold.revenue_by_date.pickup_date IS 'Calendar date of the trip pickup, derived by truncating tpep_pickup_datetime to date.';
COMMENT ON COLUMN nyc_mobility.gold.revenue_by_date.total_revenue IS 'Sum of total_amount across all Silver trips that started on this date.';
COMMENT ON COLUMN nyc_mobility.gold.revenue_by_date.trip_count IS 'Number of Silver trips that contributed to this date revenue total.';

### `nyc_mobility.gold.trips_by_borough`
Pickup-borough aggregation of Silver trips with total_revenue and trip_count per pickup_borough. Liquid clustered on pickup_borough.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.gold.trips_by_borough IS 'Pickup-borough aggregation of Silver trips with total_revenue and trip_count per pickup_borough. Liquid clustered on pickup_borough. Owner: 404Learning Data Engineering. Freshness: Downstream of Silver refresh; rebuilt as part of the Gold pipeline run.';

In [0]:
%sql
COMMENT ON COLUMN nyc_mobility.gold.trips_by_borough.pickup_borough IS 'Borough where the trip began, carried over from the Silver trips layer.';
COMMENT ON COLUMN nyc_mobility.gold.trips_by_borough.trip_count IS 'Number of Silver trips that started in this borough.';
COMMENT ON COLUMN nyc_mobility.gold.trips_by_borough.total_revenue IS 'Sum of total_amount across all Silver trips that started in this borough.';

### `nyc_mobility.gold.trips_by_hour_weekend`
Hour-of-day by weekend-flag aggregation of Silver trips with trip_count per pickup_hour and is_weekend combination. Liquid clustered on pickup_hour.

In [0]:
%sql
COMMENT ON TABLE nyc_mobility.gold.trips_by_hour_weekend IS 'Hour-of-day by weekend-flag aggregation of Silver trips with trip_count per pickup_hour-and-is_weekend combination. Liquid clustered on pickup_hour. Owner: 404Learning Data Engineering. Freshness: Downstream of Silver refresh; rebuilt as part of the Gold pipeline run.';

In [0]:
%sql
COMMENT ON COLUMN nyc_mobility.gold.trips_by_hour_weekend.pickup_hour IS 'Hour of pickup in 24-hour format from 0 to 23.';
COMMENT ON COLUMN nyc_mobility.gold.trips_by_hour_weekend.is_weekend IS 'Boolean flag that is true when the pickup occurred on Saturday or Sunday.';
COMMENT ON COLUMN nyc_mobility.gold.trips_by_hour_weekend.trip_count IS 'Number of Silver trips that fell in this hour and weekend flag bucket.';